# 2D Parallelism, Tensor plus Data

## TLDR

Sharding in notebook 02 split the model state across data-parallel GPUs, but
every GPU still ran the full width of each layer. When a single layer is too wide
for one GPU, or when communication starts to dominate, you split the layer itself
with tensor parallelism. Combined with data parallelism this is 2D parallelism.
You run it two ways here, PyTorch DTensor with FSDP and DeepSpeed AutoTP, on
Qwen2.5-0.5B across a 2 by 1 grid of two T4 GPUs. The same `TorchTrainer`
launches both.


## Introduction

Tensor parallelism splits the weight matrices inside a layer across GPUs. The
GPUs in a tensor-parallel group all see the same input, each computes part of the
layer, and they combine results with a single all-reduce. This lets a layer that
is too wide for one GPU run across several.

Data parallelism, which you have used since notebook 01, replicates or shards the
model across groups that each process different data. Put the two together and
you get a grid. Rows are tensor-parallel groups that share data and split the
model. Columns are data-parallel groups that see different data and synchronize
gradients. With tensor-parallel size 2 and data-parallel size 2 you use all two
GPUs in a 2 by 1 mesh.

We train Qwen2.5-0.5B because its attention has 2 key-value heads, and tensor
parallel size must divide that count, so 2 is the natural choice on two GPUs.
One honest note about this hardware. Tensor parallelism wants a fast link like
NVLink, because it communicates after every layer. Our T4s talk over PCIe, so we
keep tensor-parallel size small. That tradeoff is the whole lesson of when to use
which axis.


## Key concepts used in this notebook

**Tensor parallelism.** Split a weight matrix across GPUs. Column parallel splits
the output dimension, row parallel splits the input dimension. A column then row
pairing needs only one all-reduce per layer.

**2D device mesh.** A grid with a `dp` dimension and a `tp` dimension. From a
worker's rank, `tp_rank = rank % tp_size` and `dp_rank = rank // tp_size`.

**TP-aware data loading.** Every tensor-parallel rank in a data-parallel group
must see the identical batch. So you shard data by `dp_rank` and `dp_size`, not by
world rank. The global batch is `batch_per_gpu` times `dp_size`.

**DTensor and `parallelize_module`.** PyTorch's native tensor parallel API. You
give it a plan that maps each projection to column or row parallel, and it
shards the weights.

**DeepSpeed AutoTP.** Config-driven tensor parallelism. Set `autotp_size` and
DeepSpeed shards the linear layers automatically, no per-layer plan needed.


## What you will learn

- How column and row tensor parallelism split a layer with one all-reduce
- How to build a 2D device mesh and compute tp_rank and dp_rank
- Why data must be sharded by dp_rank, and what breaks if it is not
- How to apply DTensor tensor parallelism and compose it with FSDP
- How DeepSpeed AutoTP reaches 2D parallelism from a config
- When to reach for tensor parallelism versus more sharding, tied to bandwidth


## Why Ray on Anyscale for 2D parallelism

| Challenge | Without Ray | With Ray |
|---|---|---|
| Place a 2D worker grid | Hand-assign ranks to nodes and devices | `ScalingConfig(num_workers=tp*dp)`, Ray places them |
| Build process groups | Manual `new_group` wiring per topology | Ray sets up the base group, you slice the mesh |
| Two TP engines | Separate launchers | Same `train_loop_per_worker`, swap the body |
| Keep TP groups on fast links | Manual node pinning | Ray and Anyscale schedule onto the right nodes |
| Scale the grid | Rewrite launch | Change tp_size and dp_size |


## 2x2 Architecture (extension of this notebook)

A 2 by 2 mesh on four GPUs. Rows share data and split the model. Columns see
different data and synchronize gradients.

```
                 tp dimension (split the model)
                  tp_rank 0     tp_rank 1
                +------------+------------+
  dp_rank 0     |   GPU 0    |   GPU 1    |   <- TP group A, same batch
                +------------+------------+
  dp_rank 1     |   GPU 2    |   GPU 3    |   <- TP group B, same batch
                +------------+------------+
                  data-parallel groups (different batches,
                  gradients synchronized down each column)
```

The figure below shows the dataflow inside one tensor-parallel layer, and how it
composes with FSDP sharding along the data-parallel dimension.


![Tensor parallelism with FSDP](images/tp_fsdp.png)
_____________________________________________________________________________________________________________________________________
Column parallel splits a weight by output columns, row parallel splits the next
weight by input rows. The column then row pairing means the partial results only
need a single all-reduce to become the correct layer output. When FSDP is added,
the tensor-parallel shard is sharded again along the data-parallel dimension and
gathered just in time.

The next figure shows the same column then row idea on a pair of linear layers.
The first layer is split by columns and needs no communication, the second is
split by rows and finishes with one all-reduce.

![Column and row partitioning of two linear layers](images/tp_colrow.png)


## How this scales on Anyscale

| | This notebook | Production |
|---|---|---|
| Mesh | 2 TP by 1 DP on 2 T4 | For example 8 TP by many DP across nodes |
| Link for TP | PCIe, so TP size stays 2 | NVLink inside a node, TP size 8 |
| Model | Qwen2.5-0.5B | 7B to 70B and beyond |
| Change needed | -- | Raise tp_size and dp_size, keep tp inside a node |


## Cell 1 — Connect

**What you do.** Connect to Ray with the standard runtime environment.

**What to check.** Two GPUs, which we will arrange as a 2 by 1 mesh.

**Why it matters.** The two workers Ray launches become the two cells of the
mesh. Ray handles placement, you handle the math.


In [ ]:
import os
import subprocess
os.environ["RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO"] = "0"
os.environ["HF_HUB_OFFLINE"] = "1"       # 0% Hugging Face: never reach the Hub at runtime
os.environ["HF_DATASETS_OFFLINE"] = "1"

import ray
from common import utils

if not ray.is_initialized():
    ray.init(address="auto", runtime_env=utils.build_runtime_env())

utils.print_cluster_resources()

# 0% Hugging Face. Stage Qwen2.5-0.5B and WikiText from the tutorial\'s PUBLIC S3
# mirror (unsigned) into shared cluster storage once. The training workers below
# load the model and the TP-aware dataloader from these local paths.
MIRROR = "s3://anyscale-public-materials-use2/ray_summit_foundation_model_training_2026"
MODEL_NAME = "/mnt/cluster_storage/models/qwen2.5-0.5b"   # local dir, not the HF id
WIKITEXT_DIR = "/mnt/cluster_storage/datasets/wikitext-2-raw-v1"
for src, dst in [(f"{MIRROR}/models/qwen2.5-0.5b", MODEL_NAME),
                 (f"{MIRROR}/datasets/wikitext-2-raw-v1", WIKITEXT_DIR)]:
    subprocess.run(["aws", "s3", "sync", src, dst,
                    "--no-sign-request", "--only-show-errors"], check=True)

TP_SIZE, DP_SIZE = 2, 1

## TP-aware data loading

This is the detail that's easy to mess up. With tensor parallelism, the GPUs in one
tensor-parallel group hold different parts of the same model and must process the
**same** batch. If you shard data by world rank, each of those GPUs gets a
different batch and the gradients are meaningless.

So you shard by `dp_rank` over `dp_size`. The helper in `common.utils` does
exactly that with a `DistributedSampler(num_replicas=dp_size, rank=dp_rank)`.
Because each data-parallel group consumes one batch per step, the global batch
size is `batch_per_gpu` times `dp_size`, not times the world size.


## Cell 2 — The DTensor 2D training loop

**What you do.** Define a loop that builds a 2D mesh, applies DTensor tensor
parallelism to each transformer layer with a column and row plan, composes FSDP
along the data-parallel dimension, and trains.

**What to check.** The TP plan maps q, k, v, gate, and up projections to column
parallel and the o and down projections to row parallel. The model loads in fp32
and FSDP computes in fp16. Loading in fp32 keeps a master copy and avoids the
fp16 NaN you would hit by loading the model directly in fp16 on a T4.

**Why it matters.** This is real 2D parallelism. Each rank ends up holding a
fraction of the parameters that reflects both the tensor split and the FSDP
shard.


In [ ]:
import ray.train
import ray.train.torch

def dtensor_2d_train_loop(config):
    import torch
    from torch.distributed.device_mesh import init_device_mesh
    from torch.distributed.tensor.parallel import (
        ColwiseParallel, RowwiseParallel, parallelize_module,
    )
    from torch.distributed._composable.fsdp import MixedPrecisionPolicy, fully_shard
    from transformers import AutoModelForCausalLM
    import ray.train, ray.train.torch
    from common import utils

    world_rank = ray.train.get_context().get_world_rank()
    world_size = ray.train.get_context().get_world_size()
    device = ray.train.torch.get_device()
    tp_size, dp_size = config["tp_size"], config["dp_size"]
    tp_rank, dp_rank = world_rank % tp_size, world_rank // tp_size
    compute_dtype = torch.float16   # T4 has fp16 tensor cores, no native bf16

    # CHANGE 1 (vs 1D FSDP): the mesh gains a second dimension. 1D was (world_size,);
    # now it's (dp_size, tp_size). From a worker's world rank you derive its grid
    # coordinates: tp_rank = rank % tp_size (column), dp_rank = rank // tp_size (row).
    # Slice out a tp sub-mesh and a dp sub-mesh — each axis gets its own mesh.
    mesh = init_device_mesh("cuda", (dp_size, tp_size), mesh_dim_names=("dp", "tp"))
    tp_mesh, dp_mesh = mesh["tp"], mesh["dp"]

    # Load in fp32 so FSDP keeps an fp32 master copy. fp16 is for compute only.
    model = AutoModelForCausalLM.from_pretrained(config["model_name"]).to(device)
    layers = model.model.layers

    # Column-wise projections "fan out": full hidden states -> split features.
    # Row-wise projections "fan in": split features -> full hidden states.
    # This pairing avoids an extra all-gather between consecutive projections.
    tp_plan = {
        # Attention makes Q, K, and V from the same hidden states.
        # TP splits each projection by output features.
        "self_attn.q_proj": ColwiseParallel(),  # Make query chunks.
        "self_attn.k_proj": ColwiseParallel(),  # Make key chunks.
        "self_attn.v_proj": ColwiseParallel(),  # Make value chunks.

        # Attention output is built from TP partials, then folded back to hidden size.
        # TP splits this layer by input features.
        "self_attn.o_proj": RowwiseParallel(),  # Combine attention output.

        # MLP first expands hidden size to a larger intermediate size.
        # TP splits these expand layers by output features.
        "mlp.gate_proj": ColwiseParallel(),  # Make gate chunks.
        "mlp.up_proj": ColwiseParallel(),  # Make up-projection chunks.

        # MLP then compresses back to hidden size.
        # TP splits this layer by input features.
        "mlp.down_proj": RowwiseParallel(),  # Combine MLP output.
    }

    # CHANGE 2 (NEW in 2D — 1D FSDP never did this): apply tensor parallelism.
    # In 1D, every GPU held each layer at FULL width. parallelize_module with a
    # Colwise/Rowwise plan splits each layer's weight matrices ACROSS the tp sub-mesh,
    # so a layer too wide for one GPU runs across tp_size GPUs. Colwise "fans out"
    # (full hidden -> split features), Rowwise "fans in" and ends the pair with one
    # all-reduce. This happens BEFORE fully_shard.
    for layer in layers:
        parallelize_module(layer, tp_mesh, tp_plan)

    # Compose FSDP along the data-parallel dimension, computing in fp16.
    mp_policy = MixedPrecisionPolicy(param_dtype=compute_dtype, reduce_dtype=compute_dtype)

    # CHANGE 3 (vs 1D FSDP): fully_shard's mesh was the whole world; now it's dp_mesh.
    # FSDP shards ONLY along the data-parallel dimension, composing on top of the
    # tensor split from CHANGE 2 — each already-TP-split shard is sharded again over
    # dp. This composes cleanly because both TP (parallelize_module) and FSDP
    # (fully_shard) are built on DTensor.
    for layer in layers:
        fully_shard(layer, mesh=dp_mesh, mp_policy=mp_policy)  # Layer-level FSDP handles the repeated blocks.
    fully_shard(model, mesh=dp_mesh, mp_policy=mp_policy)  # Model-level FSDP handles the wrapper and any leftover parameters.

    full_params = sum(p.numel() for p in model.parameters())
    local_params = sum(
        (p.to_local().numel() if hasattr(p, "to_local") else p.numel())
        for p in model.parameters()
    )

    # DTensor does not support the fused optimizer path, so foreach is off.
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], foreach=False)

    # CHANGE 4 (vs 1D FSDP's get_dataset_shard): shard data by dp_rank/dp_size, NOT
    # world rank. Every TP rank in the same dp group MUST see the identical batch —
    # they hold different slices of one model, so a different batch per slice makes
    # the gradients meaningless. Global batch = batch_per_gpu * dp_size (not * world).
    dataloader = utils.build_causal_lm_dataloader(
        config["model_name"], dp_rank=dp_rank, dp_size=dp_size,
        seq_len=config["seq_len"], batch_size=config["batch_per_gpu"],
    )

    model.train()
    running, steps = 0.0, 0
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        loss = model(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
            labels=batch["labels"], use_cache=False,
        ).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item()
        steps += 1
        if steps >= config["max_steps"]:
            break

    import tempfile, json, os
    from ray.train import Checkpoint
    with tempfile.TemporaryDirectory() as d:
        if world_rank == 0:
            with open(os.path.join(d, "summary.json"), "w") as f:
                json.dump({"steps": steps}, f)
        ckpt = Checkpoint.from_directory(d) if world_rank == 0 else None
        ray.train.report(
            {
                "loss": running / max(steps, 1),
                "full_params": full_params,
                "local_shard": local_params,
                "shard_fraction": round(local_params / full_params, 3),
                "tp_rank": tp_rank, "dp_rank": dp_rank,
            },
            checkpoint=ckpt,
        )


## Cell 3 — Launch DTensor 2D

**What you do.** Run the DTensor loop on the 1 mesh. The number of workers
must equal tp_size times dp_size.

**What to check.** The loss is a real number, not NaN, and `shard_fraction` is
well below 1. Each rank holds only a slice of the model, split by both tensor
parallelism and FSDP.

**Why it matters.** This is the payoff. The model is split two ways at once, and
the loss is stable because we kept fp32 master weights.


In [ ]:
from ray.train import ScalingConfig, RunConfig
from ray.train.torch import TorchTrainer

dtensor_trainer = TorchTrainer(
    train_loop_per_worker=dtensor_2d_train_loop,
    train_loop_config={
        "model_name": MODEL_NAME, "tp_size": TP_SIZE, "dp_size": DP_SIZE,
        "lr": 1e-5, "seq_len": 128, "batch_per_gpu": 1, "max_steps": 10,
    },
    scaling_config=ScalingConfig(num_workers=TP_SIZE * DP_SIZE, use_gpu=True),
    run_config=RunConfig(storage_path="/mnt/cluster_storage/ray_summit_2d",
                         name="dtensor_2d"),
)
dtensor_result = dtensor_trainer.fit()
print("DTensor 2D metrics:", dtensor_result.metrics)


## The other engine, DeepSpeed AutoTP

DeepSpeed reaches tensor parallelism from a config instead of a per-layer plan.
You set the `tensor_parallel` config with an `autotp_size` and DeepSpeed finds
the linear layers and shards them. You still build the tensor-parallel and
data-parallel process groups and hand DeepSpeed a small object, the model
parallel unit, that describes the topology so gradients are synchronized only
across the data-parallel groups.

One real difference from the DTensor path. AutoTP supports ZeRO stages 1 and 2,
which shard the optimizer and gradients but not the parameters. So each rank
holds more of the parameters than in the DTensor and FSDP path, where parameters
are sharded too. You will see that in the shard fraction below. If full parameter
sharding matters, the DTensor and FSDP path is the better fit.


## Cell 4 — The DeepSpeed AutoTP training loop

**What you do.** Define a loop that builds the process groups, hands DeepSpeed a
model parallel unit, and turns on AutoTP through the config.

**What to check.** AutoTP is just the `tensor_parallel` block in the config. The
sharding happens inside `deepspeed.initialize`. DeepSpeed runs fp16 with a
dynamic loss scaler, which is why loading the model in fp16 here is stable.

**Why it matters.** Same 2D result, config-driven. For models DeepSpeed supports
out of the box, you skip writing a tensor-parallel plan entirely.


In [ ]:
def autotp_2d_train_loop(config):
    import torch, deepspeed
    import torch.distributed as dist
    from transformers import AutoModelForCausalLM
    import ray.train, ray.train.torch
    from common import utils

    world_rank = ray.train.get_context().get_world_rank()
    tp_size, dp_size = config["tp_size"], config["dp_size"]
    tp_rank, dp_rank = world_rank % tp_size, world_rank // tp_size

    # CHANGE 1 (vs 1D ZeRO): in 1D, DeepSpeed's world WAS the data-parallel group and
    # you built nothing. In 2D you construct the grid yourself: tp groups are the ROWS
    # (consecutive ranks d*tp .. (d+1)*tp — GPUs that split one model), dp groups are
    # the COLUMNS (strided ranks t, t+tp, ... — GPUs that sync gradients).
    tp_group = dp_group = None
    for d in range(dp_size):
        ranks = list(range(d * tp_size, (d + 1) * tp_size))  # TP ranks in one model split.
        g = dist.new_group(ranks)                            # Create this TP group.
        if world_rank in ranks:                              # Keep the group this rank belongs to.
            tp_group = g
    for t in range(tp_size):
        ranks = [t + d * tp_size for d in range(dp_size)]    # DP replicas of the same TP slice.
        g = dist.new_group(ranks)                            # Create this DP group.
        if world_rank in ranks:                              # Keep the group this rank belongs to.
            dp_group = g

    # CHANGE 2 (NEW in 2D): hand DeepSpeed a model-parallel unit. The MPU describes
    # the topology — which group is data-parallel, which is model(tensor)-parallel,
    # and their sizes/ranks. Passed via mpu= to initialize(), it tells DeepSpeed to
    # all-reduce gradients ONLY across the dp groups and treat the tp groups as
    # model-parallel. 1D had no MPU because there was no model-parallel dimension.
    class MPU:
        def get_data_parallel_group(self): return dp_group
        def get_model_parallel_group(self): return tp_group
        def get_data_parallel_world_size(self): return dp_size
        def get_model_parallel_world_size(self): return tp_size
        def get_data_parallel_rank(self): return dp_rank
        def get_model_parallel_rank(self): return tp_rank

    deepspeed.init_distributed()
    device = torch.device(f"cuda:{world_rank % torch.cuda.device_count()}")
    model = AutoModelForCausalLM.from_pretrained(config["model_name"], torch_dtype=torch.float16).to(device)
    full_params = sum(p.numel() for p in model.parameters())

    # CHANGE 3 (vs 1D ZeRO config): add "tensor_parallel": {"autotp_size": tp_size}
    # plus "data_parallel_size": dp_size. Just as sharding in 02 was "a stage in the
    # config," 2D here is "a tensor_parallel block in the config" — DeepSpeed finds
    # the linear layers and shards them across tp, no per-layer plan needed.

    # CHANGE 4 (vs 1D's ZeRO-3): AutoTP supports only ZeRO stages 1 and 2. Stage 1
    # shards optimizer state but NOT gradients or parameters — which is why this run's
    # shard_fraction is HIGHER than the DTensor+FSDP path (where params are sharded
    # too). If full parameter sharding matters, prefer the DTensor path.
    ds_config = {
        "train_batch_size": config["batch_per_gpu"] * dp_size,
        "train_micro_batch_size_per_gpu": config["batch_per_gpu"],
        "gradient_clipping": 1.0,
        "zero_optimization": {"stage": 1, "overlap_comm": True},
        "tensor_parallel": {"autotp_size": tp_size},   # this line turns on AutoTP
        "data_parallel_size": dp_size,
        "zero_allow_untested_optimizer": True,
        "fp16": {"enabled": True},
    }
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"])
    engine, _, _, _ = deepspeed.initialize(
        model=model, optimizer=optimizer, config=ds_config, mpu=MPU(),
    )
    local_params = sum(p.numel() for p in engine.module.parameters())

    # CHANGE 5 (same as the DTensor loop): shard data by dp_rank/dp_size, not world
    # rank — every TP rank in a dp group must see the identical batch. Global batch
    # = batch_per_gpu * dp_size.
    dataloader = utils.build_causal_lm_dataloader(
        config["model_name"], dp_rank=dp_rank, dp_size=dp_size,
        seq_len=config["seq_len"], batch_size=config["batch_per_gpu"],
    )

    engine.train()
    running, steps = 0.0, 0
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        loss = engine(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
            labels=batch["labels"], use_cache=False,
        ).loss
        engine.backward(loss)
        engine.step()
        running += loss.item()
        steps += 1
        if steps >= config["max_steps"]:
            break

    import tempfile, json, os
    from ray.train import Checkpoint
    with tempfile.TemporaryDirectory() as d:
        if world_rank == 0:
            with open(os.path.join(d, "summary.json"), "w") as f:
                json.dump({"steps": steps}, f)
        ckpt = Checkpoint.from_directory(d) if world_rank == 0 else None
        ray.train.report({
            "loss": running / max(steps, 1),
            "full_params": full_params,
            "local_shard": local_params,
            "shard_fraction": round(local_params / full_params, 3),
        }, checkpoint=ckpt)


## Cell 5 — Launch DeepSpeed AutoTP

**What you do.** Run the AutoTP loop on the same 2 by 1 grid.

**What to check.** It trains with a stable loss. The shard fraction is higher
than the DTensor run, because AutoTP with ZeRO-1 shards the optimizer but not the
parameters.

**Why it matters.** Two engines, same Ray Train surface, one clear tradeoff to
remember about parameter sharding.


In [ ]:
autotp_trainer = TorchTrainer(
    train_loop_per_worker=autotp_2d_train_loop,
    train_loop_config={
        "model_name": MODEL_NAME, "tp_size": TP_SIZE, "dp_size": DP_SIZE,
        "lr": 1e-5, "seq_len": 128, "batch_per_gpu": 1, "max_steps": 10,
    },
    scaling_config=ScalingConfig(num_workers=TP_SIZE * DP_SIZE, use_gpu=True),
    run_config=RunConfig(storage_path="/mnt/cluster_storage/ray_summit_2d",
                         name="autotp_2d"),
)
autotp_result = autotp_trainer.fit()
print("AutoTP 2D metrics:", autotp_result.metrics)


## When to use tensor parallelism

Tensor parallelism and sharding both reduce memory per GPU, but they communicate
very differently, and that decides when to use each.

| | Tensor parallelism | FSDP or ZeRO sharding |
|---|---|---|
| What moves on the wire | Activations, an all-reduce every layer | Parameters, gathered per layer |
| Volume grows with | Batch size and sequence length | Model size, fixed per step |
| On the critical path | Yes, after each layer | Overlaps with compute |
| Sensitive to latency | Very, prefers NVLink | Less, tolerates the network |
| Best placement | Inside a node on a fast link | Across nodes |

The common recipe puts tensor parallelism inside a node where NVLink is fast, and
data parallelism with sharding across nodes where communication can overlap with
compute. 

## Three additional axes: pipeline, sequence/context, and expert parallelism

We ran data, tensor, and sharded parallelism on two T4s. Two additional axes
need far more hardware to be worth it, so we cover them as concepts here and
reason about them with the planner in notebook 05.

**Pipeline parallelism** splits the model into stages of consecutive layers, one
stage per GPU. A batch flows through the stages like an assembly line. The naive
version leaves GPUs idle in a bubble while the pipeline fills and drains, so you
split each batch into micro-batches that keep every stage busy. Llama 3 and
DeepSeek-V3 push this further with schedules that overlap the forward and
backward passes.

```
   stage 0 (GPU 0)  layers 0 to 7
   stage 1 (GPU 1)  layers 8 to 15      micro-batches flow left to right,
   stage 2 (GPU 2)  layers 16 to 23     filling the pipeline to hide the bubble
   stage 3 (GPU 3)  layers 24 to 31
```

**Sequence or context parallelism** splits the sequence length across GPUs, which is useful for very long-context training where the activations and attention computation become too large for one GPU. The GPUs exchange pieces of the sequence, often with all-to-all or related collectives, so each rank can compute attention over the needed context.

**Expert parallelism** is for mixture-of-experts models. Each layer has many
expert sub-networks but a router sends each token to only a few. You place
different experts on different GPUs and route tokens to them with an all-to-all.
The model has many more parameters but each token uses only a few, so capacity
grows much faster than compute. 

## Native tensor and pipeline parallelism with Megatron on Ray

For production training that leans heavily on tensor and pipeline parallelism,
NVIDIA's Megatron is the reference implementation, and Anyscale supports running
it under Ray Train through Megatron-Bridge. The Anyscale tutorial fine-tunes
Qwen2.5-1.5B on 8 GPUs with Ray Train orchestrating Megatron-Core, so you get
Megatron's native parallelism with the same Ray scheduling and fault tolerance
you have used all course. See the Anyscale guide at
https://docs.anyscale.com/tutorials/train-with-megatron for the full walkthrough.


## Conclusion

You ran 2D parallelism two ways on a 2 by 1 grid of two T4s. DTensor split each
layer with a column and row plan and composed with FSDP, so each rank held a
small fraction of the model. DeepSpeed AutoTP reached the same grid from a
config, with a higher shard fraction because ZeRO-1 does not shard parameters.
Both fed from a TP-aware dataloader that shards by dp_rank, and both ran under the
same `TorchTrainer`.

Ray and PyTorch primitives you used. `init_device_mesh`, `parallelize_module`
with `ColwiseParallel` and `RowwiseParallel`, `fully_shard` on the dp mesh,
DeepSpeed AutoTP with a model parallel unit, and the unchanged `TorchTrainer` and
`ScalingConfig`.

You can now combine data, tensor, and sharded parallelism. Pipeline, sequence/context, and expert
parallelism are the remaining axes. In a real run all of these stack, and things
fail at scale. Notebook 04 makes the training fault tolerant and shows how to see
what every GPU is doing.
